# Assistente medico via LangGraph (Step 6) no Google Colab

Este notebook roda o fluxo LangGraph do assistente medico no Colab. Usa o mesmo modelo fine-tunado (PEFT/QLoRA) do projeto.

Antes de rodar:
1. Ative a GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU.
2. O modelo fine-tunado deve estar em outputs/finetune_pqal (clone ou Drive).
3. Clone o repositorio na celula 2.

## 1. Verificar GPU e montar Drive (opcional)

In [1]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu.returncode != 0:
    print("Aviso: GPU nao detectada. Ative em Runtime -> Change runtime type -> GPU.")
else:
    print(gpu.stdout)

from google.colab import drive
drive.mount("/content/drive")

GPU 0: Tesla T4 (UUID: GPU-746d5b06-699a-06b4-2424-d0ffabfb9423)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Clonar repositorio e instalar dependencias

In [3]:
!cd /
!rm -rf /content/POSTECH-FIAP-FASE3

In [2]:
!git clone https://github.com/thiagonespereira/POSTECH-FIAP-FASE3.git /content/POSTECH-FIAP-FASE3
%cd /content/POSTECH-FIAP-FASE3
!pip install -q -r requirements.txt
print("Repositorio e dependencias prontos.")

Cloning into '/content/POSTECH-FIAP-FASE3'...
remote: Enumerating objects: 194, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 194 (delta 74), reused 173 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (194/194), 22.95 MiB | 20.75 MiB/s, done.
Resolving deltas: 100% (74/74), done.
/content/POSTECH-FIAP-FASE3
Repositorio e dependencias prontos.


## 3. Caminho do modelo e compilar o grafo

Ajuste MODEL_DIR se o modelo estiver no Drive.

In [3]:
import sys
from pathlib import Path

REPO = Path("/content/POSTECH-FIAP-FASE3")
sys.path.insert(0, str(REPO))

MODEL_DIR = REPO / "outputs" / "finetune_pqal"
# MODEL_DIR = Path("/content/drive/MyDrive/POSTECH-FIAP-FASE3/outputs/finetune_pqal")

if not MODEL_DIR.exists():
    print("Aviso:", MODEL_DIR, "nao existe.")
else:
    print("Modelo:", MODEL_DIR)

Modelo: /content/POSTECH-FIAP-FASE3/outputs/finetune_pqal


In [4]:
from src.graphs.medical_flow import build_medical_graph

print("Carregando modelo e compilando grafo...")
app = build_medical_graph(MODEL_DIR, max_new_tokens=256)
print("Grafo compilado.")

Carregando modelo e compilando grafo...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Grafo compilado.


/content/POSTECH-FIAP-FASE3/src/models/load_llm_for_langchain.py:114: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  return HuggingFacePipeline(pipeline=pipe)


## 4. (Opcional) Ver estrutura do grafo em ASCII

In [5]:
print(app.get_graph().draw_ascii())

      +-----------+      
      | __start__ |      
      +-----------+      
            *            
            *            
            *            
+----------------------+ 
| classificar_intencao | 
+----------------------+ 
            *            
            *            
            *            
  +-----------------+    
  | buscar_contexto |    
  +-----------------+    
            *            
            *            
            *            
   +----------------+    
   | gerar_resposta |    
   +----------------+    
            *            
            *            
            *            
      +---------+        
      | validar |        
      +---------+        
            *            
            *            
            *            
        +-----+          
        | log |          
        +-----+          
            *            
            *            
            *            
      +---------+        
      | __end__ |        
      +-----

## 5. Perguntar ao assistente

In [10]:
pergunta = "Does aspirin reduce fever?"
contexto = ""

estado_inicial = {"pergunta": pergunta, "contexto": contexto or ""}
resultado = app.invoke(estado_inicial)

print("--- Resposta ---")
print(resultado.get("resposta", ""))
print("\n--- Historico ---")
for i, h in enumerate(resultado.get("historico") or []):
    content = getattr(h, "content", h) if not isinstance(h, dict) else h.get("content", h)
    print(i+1, content)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Resposta ---
Based on the following medical literature abstracts, answer the clinical question. Provide a concise evidence-based answer and state your decision: yes, no, or maybe.

Question: Does aspirin reduce fever?

Abstracts:
(No additional context provided.)

Decision: no (Resposta sem contexto adicional.)

---
**Aviso:** Esta resposta é gerada por um modelo de linguagem fine-tunado em literatura médica (PubMedQA). Ela não substitui avaliação por um profissional de saúde. Consulte sempre um médico para decisões clínicas.

--- Historico ---
1 classificar_intencao -> clinica
2 buscar_contexto (stub)
3 gerar_resposta
4 validar -> valido=True
5 log -> pergunta_len=26 resposta_len=518


## 6. Outro exemplo

In [11]:
pergunta2 = "Can the PHQ-9 assess depression in people with vision loss?"
contexto2 = "The PHQ-9 was completed by 103 participants with low vision."
resultado2 = app.invoke({"pergunta": pergunta2, "contexto": contexto2})
print(resultado2.get("resposta", ""))

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Based on the following medical literature abstracts, answer the clinical question. Provide a concise evidence-based answer and state your decision: yes, no, or maybe.

Question: Can the PHQ-9 assess depression in people with vision loss?

Abstracts:
The PHQ-9 was completed by 103 participants with low vision.

Participants were asked to rate their symptoms of depression using the PHQ-9.
The PHQ-9 is not used for assessing depression in people with vision loss.

Decision: no Evidenced Answer: The PHQ-9 can be used to assess depression in people with vision loss, but it should not be used as an alternative to other diagnostic tools like the MMSE (Mini-Mental State Examination) or the VAS (Visual Acuity Scale). It's important to note that the PHQ-9 is primarily designed for use in individuals without visual impairment, so its results may not accurately reflect the severity of depression experienced by those with vision loss. However, if you are considering using this tool, it would be bes